---
# Bagian A — Setup & Preprocessing
Mount Google Drive, ekstrak dataset, lalu audit & konversi anotasi COCO ke format YOLO.

## Import Library
Mengimpor seluruh library yang dibutuhkan untuk tahap preprocessing: manipulasi file (`shutil`, `pathlib`, `zipfile`, `os`), parsing anotasi (`json`), struktur konfigurasi (`dataclasses`), serta visualisasi (`matplotlib`, `PIL`).

In [ ]:
import json
import os
import random
import shutil
import zipfile
from collections import Counter
from dataclasses import dataclass, field
from pathlib import Path
from typing import Dict, List, Optional

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

## A.2 Konfigurasi (`Config`)

In [ ]:
@dataclass
class Config:
    # --- Path dataset ---
    dataset_root=Path(r"D:\lomba\Telepati 8.0 Datasets")
    yolo_out = Path(r"D:\lomba\train")
    coco_annotation_file= "_annotations.coco.json"

    # --- Split dataset ---
    coco_splits: List[str] = field(default_factory=lambda: ["train", "valid", "test"])
    split_mapping: Dict[str, str] = field(
        default_factory=lambda: {"train": "train", "valid": "val", "test": "test"}
    )

    # --- Reprodusibilitas ---
    seed= 42

    # --- Mapping & daftar kelas ---
    class_mapping: Dict[str, str] = field(
        default_factory=lambda: {
            "Bacterial leaf blight": "bacterial_leaf_blight",
            "Bacterial panicle Blight": "bacterial_panicle_blight",
            "Blast": "blast",
            "Leaf blast": "blast",
            "Infected Blast": "blast",
            "Brown spot": "brown_spot",
            "BrownSpot": "brown_spot",
            "False-Smut": "false_smut",
            "Healthy Rice Leaf": "healthy",
            "Healthy Rice beads": "healthy",
            "Healthy": "healthy",
            "healthy": "healthy",
            "Leaf-roller": "leaf_roller",
            "Leaf Scald": "leaf_scald",
            "Leaf scald": "leaf_scald",
            "Narrow brown": "narrow_brown",
            "Sheath Blight": "sheath_blight",
            "Rice-Tungro": "tungro",
        }
    )
    class_names: List[str] = field(
        default_factory=lambda: [
            "bacterial_leaf_blight",
            "bacterial_panicle_blight",
            "blast",
            "brown_spot",
            "false_smut",
            "healthy",
            "leaf_roller",
            "leaf_scald",
            "narrow_brown",
            "sheath_blight",
            "tungro",
        ]
    )

    # --- Hyperparameter training YOLO26s-p2 ---
    model_yaml= "yolo26s-p2.yaml"
    train_epochs = 300
    train_batch = 32
    train_imgsz = 640
    train_weight_decay = 0.001
    train_save_period = 50
    train_pretrained = False
    train_hsv_h = 0.01
    train_hsv_s= 0.3
    train_hsv_v = 0.3
    train_translate= 0.2
    train_shear = 0
    train_perspective = 0
    train_scale = 0.01
    train_degrees = 0
    train_fliplr = 0.3
    train_flipud = 0.3
    train_mosaic = 0.2
    train_close_mosaic = 10
    train_erasing = 0.0
    train_auto_augment = None
    train_cos_lr = False
    train_patience = 0
    train_project = Path(r"D:\lomba\train")
    train_name = "train_yolo_26n_p2"

    @property
    def class_to_id(self) -> Dict[str, int]:
        return {name: i for i, name in enumerate(self.class_names)}

    @property
    def best_weights_path(self) -> str:
        return str(Path(self.train_project) / self.train_name / "weights" / "best.pt")


CFG = Config()


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)


set_seed(CFG.seed)

## Cek Ketersediaan Folder & File Anotasi COCO
`check_dataset_structure()` memverifikasi bahwa setiap folder split beserta file `_annotations.coco.json`-nya benar-benar ada 

In [ ]:
def check_dataset_structure(cfg: Config) -> None:
    for split in cfg.coco_splits:
        split_dir = cfg.dataset_root / split
        json_path = split_dir / cfg.coco_annotation_file

        print(f"\n[{split}]")
        print("Folder :", split_dir)
        print("JSON   :", json_path)
        print("JSON exists :", json_path.exists())


check_dataset_structure(CFG)

## Audit Dataset COCO
`audit_coco_json()` membaca satu file anotasi COCO dan melaporkan jumlah gambar & anotasi, gambar tanpa anotasi, bbox yang tidak valid/keluar batas, dan distribusi kategori. `run_coco_audit()` menjalankannya untuk seluruh split sekaligus.

In [ ]:
def audit_coco_json(json_path: Path) -> dict:
    data = json.loads(json_path.read_text(encoding="utf-8"))

    images = {img["id"]: img for img in data["images"]}
    categories = {cat["id"]: cat["name"] for cat in data["categories"]}

    anotasi_per_image = {}
    for anotasi in data["annotations"]:
        anotasi_per_image.setdefault(anotasi["image_id"], []).append(anotasi)

    gambar_tanpa_anotasi = [
        gambar
        for id_gambar, gambar in images.items()
        if id_gambar not in anotasi_per_image
    ]

    invalid_bbox = []
    for anotasi in data["annotations"]:
        if anotasi["image_id"] not in images:
            continue

        img = images[anotasi["image_id"]]
        W = img["width"]
        H = img["height"]
        bbox = anotasi.get("bbox")

        if bbox is None or len(bbox) != 4 or bbox[2] <= 0 or bbox[3] <= 0:
            invalid_bbox.append(anotasi)
            continue

        x, y, w, h = bbox
        if x < 0 or y < 0 or x + w > W or y + h > H:
            invalid_bbox.append(anotasi)

    class_counts = Counter()
    for anotasi in data["annotations"]:
        nama = categories.get(anotasi["category_id"], "UNKNOWN")
        class_counts[nama] += 1

    print(f"Gambar              : {len(images)}")
    print(f"Anotasi            : {len(data['annotations'])}")
    print(f"Gambar tanpa anotasi : {len(gambar_tanpa_anotasi)}")
    print(f"Bbox invalid/outside    : {len(invalid_bbox)}")
    print("Distribusi kategori:")
    for name, count in class_counts.most_common():
        print(f"{name:<35}: {count}")

    return data


def run_coco_audit(cfg: Config) -> Dict[str, dict]:
    data_coco = {}
    for split in cfg.coco_splits:
        json_path = cfg.dataset_root / split / cfg.coco_annotation_file
        print(f"AUDIT {split.upper()}")
        data_coco[split] = audit_coco_json(json_path)
    return data_coco


data_coco = run_coco_audit(CFG)

## Membuat Struktur Folder Output YOLO
`create_yolo_dirs()` membuat folder `images/{train,val,test}` dan `labels/{train,val,test}`.

In [ ]:
def create_yolo_dirs(cfg: Config) -> None:
    for split in ["train", "val", "test"]:
        (cfg.yolo_out / "images" / split).mkdir(parents=True, exist_ok=True)
        (cfg.yolo_out / "labels" / split).mkdir(parents=True, exist_ok=True)

    print("Folder YOLO berhasil dibuat.")


create_yolo_dirs(CFG)

## Konversi Dataset COCO Ke Format YOLO
`convert_coco_to_yolo()` adalah tahap inti preprocessing: memetakan nama kelas via `cfg.class_mapping`, meng-*clip* bounding box yang keluar batas gambar, membuang bbox tidak valid, mengonversi koordinat piksel COCO `(x, y, w, h)` menjadi format YOLO ternormalisasi, lalu menyalin gambar & menulis file label yang memiliki minimal satu anotasi valid

In [ ]:
def convert_coco_to_yolo(cfg: Config) -> None:
    class_to_id = cfg.class_to_id

    for split in cfg.coco_splits:
        input_dir = cfg.dataset_root / split
        json_path = input_dir / cfg.coco_annotation_file

        output_split = "val" if split == "valid" else split
        image_out = cfg.yolo_out / "images" / output_split
        label_out = cfg.yolo_out / "labels" / output_split

        with open(json_path, "r", encoding="utf-8") as f:
            coco = json.load(f)

        images = {image["id"]: image for image in coco["images"]}
        categories = {
            category["id"]: category["name"] for category in coco["categories"]
        }

        annotations_per_image = {image_id: [] for image_id in images}

        # dipakai untuk melacak kelas apa saja yang ada di gambar yang akhirnya di-skip
        raw_categories_per_image = {image_id: [] for image_id in images}

        invalid_count = 0
        clipped_count = 0
        unmapped_count = 0

        invalid_class_counts = Counter()
        clipped_class_counts = Counter()
        unmapped_class_counts = Counter()

        for ann in coco["annotations"]:
            image_id = ann["image_id"]
            category_name = categories[ann["category_id"]]
            raw_categories_per_image[image_id].append(category_name)

            mapped_class = cfg.class_mapping.get(category_name)

            if mapped_class is None:
                unmapped_count += 1
                unmapped_class_counts[category_name] += 1
                continue

            class_id = class_to_id[mapped_class]

            x, y, w, h = ann["bbox"]
            img_w = images[image_id]["width"]
            img_h = images[image_id]["height"]

            x1, y1, x2, y2 = x, y, x + w, y + h

            new_x1 = max(0, x1)
            new_y1 = max(0, y1)
            new_x2 = min(img_w, x2)
            new_y2 = min(img_h, y2)

            new_w = new_x2 - new_x1
            new_h = new_y2 - new_y1

            if new_w <= 0 or new_h <= 0:
                invalid_count += 1
                invalid_class_counts[category_name] += 1
                continue

            if new_x1 != x1 or new_y1 != y1 or new_x2 != x2 or new_y2 != y2:
                clipped_count += 1
                clipped_class_counts[category_name] += 1

            x_center = (new_x1 + new_x2) / 2 / img_w
            y_center = (new_y1 + new_y2) / 2 / img_h
            width = new_w / img_w
            height = new_h / img_h

            annotations_per_image[image_id].append(
                f"{class_id} {x_center:.6f} {y_center:.6f} {width:.6f} {height:.6f}"
            )

        # counter untuk gambar yang dimasukkan vs di-skip
        included_images = 0
        skipped_images = 0
        skipped_no_annotation = 0  
        skipped_all_filtered = 0  
        skipped_class_counts = Counter()

        for image_id, image_info in images.items():
            if not annotations_per_image[image_id]:
                skipped_images += 1

                raw_classes = raw_categories_per_image[image_id]
                if not raw_classes:
                    skipped_no_annotation += 1
                else:
                    skipped_all_filtered += 1
                    for c in raw_classes:
                        skipped_class_counts[c] += 1
                continue

            file_name = image_info["file_name"]
            source_image = input_dir / file_name

            if not source_image.exists():
                continue

            shutil.copy2(source_image, image_out / Path(file_name).name)

            label_name = Path(file_name).stem + ".txt"
            label_path = label_out / label_name

            with open(label_path, "w") as f:
                for line in annotations_per_image[image_id]:
                    f.write(line + "\n")

            included_images += 1

        # ================= RINGKASAN =================
        print(f"\n{'='*50}")
        print(f"  SPLIT: {split.upper()}")
        print(f"{'='*50}")

        print(f"\nTotal gambar                : {len(images)}")
        print(f"Gambar dimasukkan ke YOLO   : {included_images}")
        print(f"Gambar di-skip (total)      : {skipped_images}")
        print(f"  - tanpa annotation sama sekali : {skipped_no_annotation}")
        print(f"  - semua annotation dibuang     : {skipped_all_filtered}")

        print(f"\nBbox clipped   : {clipped_count}")
        print(f"Bbox invalid   : {invalid_count}")
        print(f"Bbox unmapped  : {unmapped_count}")

        print("\n-- Kelas pada gambar yang di-skip --")
        if skipped_class_counts:
            for name, count in skipped_class_counts.most_common():
                print(f"  {name:<35}: {count}")
        else:
            print("  (tidak ada / semua gambar skip memang tanpa annotation)")

        print("\n-- Kelas & jumlah bbox invalid --")
        if invalid_class_counts:
            for name, count in invalid_class_counts.most_common():
                print(f"  {name:<35}: {count}")
        else:
            print("  (tidak ada)")

        print("\n-- Kelas & jumlah bbox clipped --")
        if clipped_class_counts:
            for name, count in clipped_class_counts.most_common():
                print(f"  {name:<35}: {count}")
        else:
            print("  (tidak ada)")

        print("\n-- Kelas & jumlah bbox unmapped (tidak ada di CLASS_MAPPING) --")
        if unmapped_class_counts:
            for name, count in unmapped_class_counts.most_common():
                print(f"  {name:<35}: {count}")
        else:
            print("  (tidak ada)")


convert_coco_to_yolo(CFG)

## Membuat File Konfigurasi `data.yaml`
`write_data_yaml()` menulis file `data.yaml` yang dibutuhkan Ultralytics: lokasi dataset, path tiap split, jumlah kelas (`nc`), dan nama-nama kelas.

In [ ]:
def write_data_yaml(cfg: Config) -> str:
    yaml_text = f"""
path: {cfg.yolo_out}

train: images/train
val: images/val
test: images/test

nc: {len(cfg.class_names)}

names:
"""
    for i, name in enumerate(cfg.class_names):
        yaml_text += f"  {i}: {name}\n"

    with open(cfg.yolo_out / "data.yaml", "w") as f:
        f.write(yaml_text)

    print(yaml_text)
    return yaml_text


write_data_yaml(CFG)

## Verifikasi Hasil Konversi Dataset
`verify_yolo_dataset()` memastikan jumlah gambar dan label pada tiap split sinkron. `show_sample_label()` menampilkan contoh isi salah satu file label untuk memastikan format YOLO sudah benar.

In [ ]:
def verify_yolo_dataset(cfg: Config) -> None:
    for split in ["train", "val", "test"]:
        images = list((cfg.yolo_out / "images" / split).glob("*"))
        labels = list((cfg.yolo_out / "labels" / split).glob("*.txt"))
        print(split, "images:", len(images), "labels:", len(labels))


def show_sample_label(cfg: Config, split: str = "train") -> None:
    label_files = list((cfg.yolo_out / "labels" / split).glob("*.txt"))
    print(label_files[0])
    print()
    print(label_files[0].read_text())


verify_yolo_dataset(CFG)
show_sample_label(CFG, split="train")

## Visualisasi Sampel Anotasi
`visualisasi_sampel()` menampilkan beberapa gambar contoh beserta bounding box hasil konversi  untuk memvalidasi secara visual

In [ ]:
def visualisasi_sampel(
    output_dir: Path, split: str = "train", sample_size: int = 20, cfg: Config = CFG
) -> None:
    images_dir = output_dir / "images" / split
    labels_dir = output_dir / "labels" / split

    image_files = [
        p
        for p in images_dir.rglob("*")
        if p.suffix.lower() in [".jpg", ".jpeg", ".png"]
    ]

    random.seed(cfg.seed)

    samples = random.sample(image_files, min(sample_size, len(image_files)))

    cols = 3
    rows = int(np.ceil(len(samples) / cols))

    fig, axes = plt.subplots(rows, cols, figsize=(15, 5 * rows))

    axes = np.array(axes).reshape(-1)

    for ax, image_path in zip(axes, samples):
        image = Image.open(image_path)

        W, H = image.size

        ax.imshow(image)

        label_path = (labels_dir / image_path.relative_to(images_dir)).with_suffix(
            ".txt"
        )

        # Kalau label tidak ada
        if not label_path.exists():
            ax.set_title(image_path.name)
            ax.axis("off")
            continue

        lines = label_path.read_text(encoding="utf-8").splitlines()

        for line in lines:
            if not line.strip():
                continue

            class_id, xc, yc, w, h = map(float, line.split())

            class_id = int(class_id)

            # YOLO normalized -> pixel
            xc *= W
            yc *= H
            w *= W
            h *= H

            x1 = xc - w / 2
            y1 = yc - h / 2

            rect = plt.Rectangle((x1, y1), w, h, fill=False, linewidth=2)

            ax.add_patch(rect)

            ax.text(
                x1, y1, cfg.class_names[class_id], fontsize=8, backgroundcolor="white"
            )

        ax.set_title(image_path.name)

        ax.axis("off")

    for ax in axes[len(samples) :]:
        ax.axis("off")

    plt.tight_layout()
    plt.show()


visualisasi_sampel(CFG.yolo_out, split="train", sample_size=20)

## Distribusi Jumlah Anotasi per Kelas
`compute_class_distribution()` menghitung dan menampilkan tabel distribusi jumlah anotasi per kelas pada tiap split (`train`/`val`/`test`) untuk mengecek keseimbangan kelas (*class imbalance*) sebelum training.

In [ ]:
def compute_class_distribution(cfg: Config):
    import pandas as pd

    distribution = {}

    for split in ["train", "val", "test"]:
        label_dir = cfg.yolo_out / "labels" / split

        counter = Counter()

        for label_path in label_dir.glob("*.txt"):
            lines = label_path.read_text(encoding="utf-8").splitlines()

            for line in lines:
                if not line.strip():
                    continue

                class_id = int(line.split()[0])

                counter[class_id] += 1

        distribution[split] = counter

    df = pd.DataFrame(
        {
            "class": cfg.class_names,
            "train": [distribution["train"][i] for i in range(len(cfg.class_names))],
            "val": [distribution["val"][i] for i in range(len(cfg.class_names))],
            "test": [distribution["test"][i] for i in range(len(cfg.class_names))],
        }
    )

    df["total"] = df["train"] + df["val"] + df["test"]

    df = df.sort_values("total", ascending=False).reset_index(drop=True)

    print(df)
    return df


class_distribution_df = compute_class_distribution(CFG)

---
# Bagian B — Training
Fase training

## Augmentasi Custom (Albumentations)
`build_custom_transforms()` mengembalikan pipeline augmentasi tambahan: kombinasi baseline (blur ringan, grayscale , CLAHE, kontras/brightness ringan, ISO noise, sharpen, kompresi gambar). Ini melengkapi augmentasi bawaan Ultralytics yang diatur langsung di parameter `train()`.

In [ ]:
import albumentations as A

def build_custom_transforms() -> list:
    return [
        A.Blur(blur_limit=(3, 5), p=0.01),
        A.MedianBlur(blur_limit=(3, 5), p=0.01),
        A.ToGray(p=0.01, method="weighted_average", num_output_channels=3),
        A.CLAHE(clip_limit=(1.0, 4.0), tile_grid_size=(8, 8), p=0.15),
        A.RandomBrightnessContrast(brightness_limit=0.1, contrast_limit=0.1, p=0.1),
        A.ISONoise(color_shift=(0.01, 0.03), intensity=(0.1, 0.3), p=0.05),
        A.Sharpen(alpha=(0.1, 0.2), lightness=(0.9, 1.0), p=0.05),
        A.ImageCompression(quality_range=(70, 95), p=0.05),
    ]
custom_transforms = build_custom_transforms()

## Training Model YOLO26n-p2
`train_model()` melatih model  menggunakan seluruh hyperparameter dari `CFG`, dikombinasikan dengan `custom_transforms` dari Albumentations. Hasil training disimpan ke `CFG.train_project/CFG.train_name`.

In [ ]:
from ultralytics import YOLO
def train_model(cfg: Config, augmentations: list):
    model = YOLO(cfg.model_yaml)
    results = model.train(
        data=str(cfg.yolo_out / "data.yaml"),
        epochs=cfg.train_epochs,
        batch=cfg.train_batch,
        imgsz=cfg.train_imgsz,
        weight_decay=cfg.train_weight_decay,
        save_period=cfg.train_save_period,
        pretrained=cfg.train_pretrained,
        seed=cfg.seed,
        # HSV 
        hsv_h=cfg.train_hsv_h,
        hsv_s=cfg.train_hsv_s,
        hsv_v=cfg.train_hsv_v,
        # Geometric
        translate=cfg.train_translate,
        shear=cfg.train_shear,
        perspective=cfg.train_perspective,
        scale=cfg.train_scale,
        degrees=cfg.train_degrees,
        # Flip
        fliplr=cfg.train_fliplr,
        flipud=cfg.train_flipud,
        # Mosaic dan erasing
        mosaic=cfg.train_mosaic,
        close_mosaic=cfg.train_close_mosaic,
        erasing=cfg.train_erasing,
        auto_augment=cfg.train_auto_augment,
        # LR schedule 
        cos_lr=cfg.train_cos_lr,
        patience=cfg.train_patience,
        augmentations=augmentations,
        project=cfg.train_project,
        name=cfg.train_name,
        exist_ok=True,
    )

    return model, results


model_yolo26n_p2, results_yolo26n_p2 = train_model(CFG, custom_transforms)

---
# Bagian C — Evaluation
Memuat bobot model terbaik hasil training, lalu mengevaluasinya pada data validation dan test.

## Load Model Terbaik Hasil Training
`load_best_model()` memuat checkpoint `best.pt` untuk dipakai pada tahap evaluasi.

In [ ]:
def load_best_model(weights_path: str) -> YOLO:
    return YOLO(weights_path)


best_model = load_best_model(CFG.best_weights_path)

## Evaluasi Model pada Data Validation
`evaluate_model()` menjalankan evaluasi resmi Ultralytics pada split tertentu dan mengembalikan objek metrik berisi mAP50, mAP50-95, Precision, dan Recall.

In [ ]:
def evaluate_model(model: YOLO, cfg: Config, split: str):
    metrics = model.val(
        data=str(cfg.yolo_out / "data.yaml"), split=split, imgsz=cfg.train_imgsz
    )
    return metrics


metrics = evaluate_model(best_model, CFG, split="val")

print("mAP50 :", metrics.box.map50)
print("mAP50-95 :", metrics.box.map)

## Evaluasi Akhir pada Data Test
Evaluasi final pada split `test` (data yang sama sekali tidak dilihat selama training/validasi).

In [ ]:
def compute_f1(precision: float, recall: float) -> float:
    if precision + recall > 0:
        return 2 * precision * recall / (precision + recall)
    return 0


final_metrics = evaluate_model(best_model, CFG, split="test")

print("Test mAP50 :", final_metrics.box.map50)
print("Test mAP50-95 :", final_metrics.box.map)

test_precision = final_metrics.box.mp
test_recall = final_metrics.box.mr
test_f1 = compute_f1(test_precision, test_recall)

print("Test Precision :", test_precision)
print("Test Recall    :", test_recall)
print("Test F1-score  :", test_f1)